# 00 · Explore the history

A first look at the historical data: what are we working with? Reads the local Parquet (`data/raw/rawtree_history`, from `python -m training.data.export_history`) through DuckDB. Views: `catalog`, `sessions` (carousels), `events` (organic activity). For single rows, use the CLI: `python -m training.data.inspect product|user|carousel <id>`.

## What we can embed: the available information

Measured on the full history (60 days, 15 Sep → 14 Nov). **Live** says whether Bas can give the model the same field at serve time.

### Product side: one row per product in `catalog` (5,000 products)

| Field | What it is | Values | Live |
|---|---|---|---|
| `product_id` | opaque ID | 5,000 | yes (new products get new IDs) |
| `category` | top-level category | 10 | yes |
| `subcategory` | finer type | 89 (nested in category) | yes |
| `brand` | brand name | 80 (a brand spans up to 9 subcategories) | yes |
| `price_eur` | price | €1.99 – €643.99, median €30.99; **constant during the history** | yes (current price) |
| `description` | 1–2 sentences of text | 4,983 unique | yes |
| `created_at` | when it was added | 29 May 2024 – 15 Sep 2026 (all before the history) | yes |
| `is_active` | in stock | always `true` in the history | Bas filters on it |

**Derived from behaviour** (history only, not from content):
- popularity: views, add-to-carts and purchases in `events`
- conversion: view → cart → purchase
- carousel CTR and purchase rate per slot
- co-occurrence: products viewed or bought together in a session, or in `recent_views`

A **new product has only the content fields**, so the product embedding must work from content alone.

### User / request side: one row per carousel in `sessions` (2,000,000 carousels, 59,698 users)

| Field | What it is | Values | Live |
|---|---|---|---|
| `user_id` | opaque ID | 59,698 (median 20 carousels each) | yes |
| `device` | web, ios, android | 3; **fixed per user** | yes |
| `region` | Spanish region | 10; **fixed per user** | yes |
| `page` | home, product, category | 3; differs per carousel | yes |
| `recent_views` | products viewed earlier in this visit, with timestamps | 0–6 (mean 2.0; 20 % empty); last view median 31 s before the carousel | yes (Bas: IDs, oldest first) |
| `ts` | time of the request | hour of day, day of week | yes |
| `is_new_user` | first visit | *not in the history*: derive it from the first time a user appears | yes |

**The user's past organic behaviour**, from `events` (12 M rows, 2.5 M visits, median 4 events per visit):
- views, add-to-carts and purchases with product, price, device and region
- 99.8 % of carousel users also appear in `events`, so most users have a history to embed

Note: `recent_views` **is available**, both in every history carousel and in every live request. What it is *not* is a part of the organic event log: a sample of recent views matched no view event in `events` with the same user, product and time. So `recent_views` (this visit) and `events` (the user's organic history) are two separate inputs, and we can't rebuild one from the other.

### Labels: what the model learns to predict

- `shown`: the 10 products, in slot order.
- `propensity`: how likely the old system was to put each one in its slot. Used to correct for its popularity bias.
- `clicked`: at most 1 product per carousel. It is always one of the shown products. 3.7 % of carousels get a click.
- `purchased`: at most 1 product per carousel. A purchased product is always the clicked one. 1.4 % of carousels end in a purchase.
- `revenue_eur`.

**What we embed, in short:**
- **Product:** category, subcategory, brand, price and description text, optionally with an ID embedding plus behaviour statistics for known products.
- **User:** device, region, the user's organic history (when known), and per request the page, time and `recent_views` (embedded with the product embedding).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root, when opened from analysis/

import matplotlib.pyplot as plt
from training.data.paths import con, history_dir

c = con()
print(history_dir())
c.sql("SELECT 'catalog' t, count(*) n FROM catalog UNION ALL SELECT 'sessions', count(*) FROM sessions UNION ALL SELECT 'events', count(*) FROM events")

## Volume per day

In [ ]:
daily = c.sql("""
  SELECT d, sum(carousels) carousels, sum(views) views, sum(carts) carts, sum(purchases) purchases FROM (
    SELECT ts::DATE d, count(*) carousels, 0 views, 0 carts, 0 purchases FROM sessions GROUP BY d
    UNION ALL
    SELECT ts::DATE, 0, count_if(event_type='view'), count_if(event_type='add_to_cart'), count_if(event_type='purchase') FROM events GROUP BY 1
  ) GROUP BY d ORDER BY d""").df()
daily.set_index('d').plot(subplots=True, figsize=(12, 8), title='per day'); plt.tight_layout()

## Context mix (device · region · page) and carousel outcomes

In [ ]:
c.sql("""SELECT device, page, count(*) carousels, round(avg((clicked IS NOT NULL)::INT), 4) ctr,
       round(avg((purchased IS NOT NULL)::INT), 4) purchase_rate
FROM sessions GROUP BY ALL ORDER BY carousels DESC""").df()

In [ ]:
c.sql("""SELECT region, count(*) carousels, round(avg((clicked IS NOT NULL)::INT), 4) ctr,
       round(avg((purchased IS NOT NULL)::INT), 4) purchase_rate FROM sessions GROUP BY ALL ORDER BY carousels DESC""").df()

## Purchases per category (organic vs. carousel)

In [ ]:
c.sql("""
WITH organic AS (SELECT c.category, count(*) n FROM events e JOIN catalog c USING (product_id)
                 WHERE event_type = 'purchase' GROUP BY ALL),
     carousel AS (SELECT c.category, count(*) n FROM sessions s JOIN catalog c ON c.product_id = s.purchased GROUP BY ALL),
     cat AS (SELECT category, count(*) products, round(median(price_eur), 2) median_price FROM catalog GROUP BY ALL)
SELECT category, products, median_price, organic.n organic_purchases, carousel.n carousel_purchases
FROM cat LEFT JOIN organic USING (category) LEFT JOIN carousel USING (category) ORDER BY organic_purchases DESC""").df()

## Prices

In [ ]:
prices = c.sql("SELECT price_eur FROM catalog").df()
bought = c.sql("SELECT price_eur FROM events WHERE event_type = 'purchase'").df()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
prices.price_eur.plot.hist(bins=60, ax=ax[0], logy=True, title='catalogue prices')
bought.price_eur.plot.hist(bins=60, ax=ax[1], logy=True, title='prices of organic purchases'); plt.tight_layout()

## Position effect in the carousel

In [ ]:
c.sql("""
WITH s AS (SELECT unnest(range(1, 11)) slot, unnest(shown) pid, clicked, purchased FROM sessions)
SELECT slot, round(avg((clicked = pid)::INT), 5) ctr, round(avg((purchased = pid)::INT), 5) purchase_rate FROM s GROUP BY slot ORDER BY slot""").df()